In [ ]:
%load_ext tensorboard
#!rm -rf ./logs/ 

import os
import pandas as pd
import numpy as np
import tensorflow as tf

from datetime import datetime
from tensorflow import keras
from tensorflow.keras import layers
from tensorboard.plugins.hparams import api as hp

import warnings
warnings.filterwarnings("ignore")

!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/models.py" -P models -nc
!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/aux_func.py" -P aux_func -nc
import sys
sys.path.append("models")
sys.path.append("aux_func")

from aux_func import *
from models import *

print(tf.version.VERSION)
print(keras.__version__)

gpus = tf.config.list_physical_devices("GPU")        
log_gpus = tf.config.list_logical_devices('GPU')
print(len(gpus), "Physical GPUs,", len(log_gpus), "Logical GPU")

from psutil import *

!lscpu|grep "Model name"
!nvidia-smi -L

In [ ]:
!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/data_set.csv" -P data_set -nc

data = pd.read_csv(
    "data-set/data-set.csv",
     sep=";")

In [ ]:
df = data.copy()

pro_max = df["item_code"].astype("int64").max()
rec_max = df["recency"].astype("int64").max()
pay_max = df["payment"].astype("int64").max()
cat_max = df["category_code"].astype("int64").max()
met_max = df["method_code"].astype("int64").max()
mon_max = df["month"].astype("int64").max()
day_max = df["dayofweek"].astype("int64").max()
n_padded = 24
batch_size = 128

In [ ]:
train_data = create_sequences(train)
train_data = create_splits(train_data)
train_list = make_padding(train_data, n_padded)

In [ ]:
MODEL_NUM = hp.HParam("model_num", hp.Discrete([8]))
RNN_STYLE = hp.HParam("rnn_style", hp.Discrete(["lstm"]))
EMB_UNITS = hp.HParam("emb_units", hp.Discrete([32, 64, 96, 128])) 
RNN_UNITS = hp.HParam("rnn_units", hp.Discrete([100, 200, 300, 400, 500, 600])) 
DROP_RATE = hp.HParam("drop_rate", hp.Discrete([0.6, 0.8]))
L_RATE = hp.HParam("l_rate", hp.Discrete([0.001]))
REPLICATION = hp.HParam("replication", hp.Discrete([1, 2]))
METRIC_ACCURACY = "sparse_categorical_accuracy"

ldir = "logs/time-tune/"
with tf.summary.create_file_writer(ldir).as_default():
    hp.hparams_config(
          hparams=[MODEL_NUM, RNN_STYLE, EMB_UNITS, RNN_UNITS,
                   DROP_RATE, L_RATE, REPLICATION],
          metrics=[hp.Metric(METRIC_ACCURACY, display_name="accuracy")])

In [ ]:
def run(run_dir, hparams, model):
    with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)
        if hparams["model"]==1:
            input_list=[train_list[0]],
        elif hparams["model"]==2:
            input_list=[train_list[0], train_list[1]]
        elif hparams["model"]==3:
            input_list=[train_list[0], train_list[1],
                        train_list[5]]
        elif hparams["model"]==4:
            input_list=[train_list[0], train_list[1],
                        train_list[5], train_list[6]]
        elif hparams["model"]==5:
            input_list=[train_list[0], train_list[1],
                        train_list[5], train_list[6],
                        train_list[2]]
        elif hparams["model"]==6:
            input_list=[train_list[0], train_list[1],
                        train_list[5], train_list[6],
                        train_list[4]]
        elif hparams["model"]==7:
            input_list=[train_list[0], train_list[1],
                        train_list[5], train_list[6],
                        train_list[3]]
        else:
            input_list=[train_list[0], train_list[1],
                        train_list[5], train_list[6],
                        train_list[2], train_list[3]]
        hist = model_training(model,
                              input_list,
                              train_list[-1],
                              val_split,
                              batch_size=batch_size)
        m = hist.history["val_sparse_categorical_accuracy"][-1]
        print("m:",m)  
        tf.summary.scalar(METRIC_ACCURACY, m, step=2)

In [ ]:
session_num = 0
for bi in [False]:
    for s in RNN_STYLE.domain.values:
        for n in MODEL_NUM.domain.values:
            for e in EMB_UNITS.domain.values:
                for g in RNN_UNITS.domain.values:
                    for d in DROP_RATE.domain.values:
                        for l in L_RATE.domain.values:
                            for r in REPLICATION.domain.values:
                                hparams = {
                                    "bidirect": bi,
                                    "style": s,
                                    "model": n,
                                    "emb_unit": e,
                                    "rnn_unit": g,
                                    "dropout": d,
                                    "learning": l,
                                    "n_padded": n_padded,
                                    "item_max": pro_max+1,
                                    "rec_max": rec_max+1,
                                    "pay_max": pay_max+1,
                                    "cat_max": cat_max+1,
                                    "met_max": met_max+1,
                                    "mon_max": mon_max+1,
                                    "day_max": day_max+1,
                                    RNN_STYLE: s,
                                    MODEL_NUM: n,
                                    EMB_UNITS: e,
                                    RNN_UNITS: g,
                                    DROP_RATE: d,
                                    L_RATE: l,
                                    REPLICATION: r,
                                }
                                model = model_base(hparams)
                                run_name = f"model{n}"+"-"+str(session_num)+"-"+str(r)
                                print("Experiment: %s" % run_name)
                                run(ldir + run_name, hparams, model)
                            session_num += 1